<a href="https://colab.research.google.com/github/arnavon2005/Army_Provost_ML_Project/blob/main/06_Decision_Support_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Decision-Support System

This notebook develops the decision-support layer of the Army Provost ML project. It builds upon the completed exploratory analysis and machine-learning model developed in the previous notebooks.

The objective is to transform incident-level predictions and historical crime patterns into interpretable information that can assist control-room personnel with incident assessment, prioritization, and response planning.

The system is a decision-support prototype. It does not autonomously make operational decisions or claim to represent official Indian Army doctrine.

In [1]:
# ============================================================
# DECISION-SUPPORT SYSTEM
# PROJECT ENVIRONMENT SETUP
# ============================================================

from google.colab import drive
import os

print("=" * 70)
print("ARMY PROVOST DECISION-SUPPORT SYSTEM")
print("PROJECT ENVIRONMENT SETUP")
print("=" * 70)

# ------------------------------------------------------------
# Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# Existing project root
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"

DATASETS_PATH = os.path.join(
    PROJECT_ROOT,
    "Datasets"
)

CLEANED_DATA_PATH = os.path.join(
    DATASETS_PATH,
    "Cleaned"
)

MODELS_PATH = os.path.join(
    PROJECT_ROOT,
    "Models"
)

FIGURES_PATH = os.path.join(
    PROJECT_ROOT,
    "Figures"
)

OUTPUTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs"
)

REPORTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Reports"
)

LOGS_PATH = os.path.join(
    PROJECT_ROOT,
    "Logs"
)

# ------------------------------------------------------------
# Important saved ML artifacts
# ------------------------------------------------------------

RF_MODEL_PATH = os.path.join(
    MODELS_PATH,
    "final_random_forest.pkl"
)

PREPROCESSING_PIPELINE_PATH = os.path.join(
    MODELS_PATH,
    "preprocessing_pipeline.pkl"
)

# ------------------------------------------------------------
# Verify project structure
# ------------------------------------------------------------

required_directories = [
    PROJECT_ROOT,
    DATASETS_PATH,
    CLEANED_DATA_PATH,
    MODELS_PATH,
    FIGURES_PATH,
    OUTPUTS_PATH,
    REPORTS_PATH,
    LOGS_PATH
]

print("\nChecking project directories...\n")

all_directories_available = True

for path in required_directories:
    exists = os.path.isdir(path)

    print(
        f"{'✓' if exists else '✗'} "
        f"{path}"
    )

    if not exists:
        all_directories_available = False

# ------------------------------------------------------------
# Verify important ML artifacts
# ------------------------------------------------------------

print("\nChecking important ML artifacts...\n")

required_files = [
    RF_MODEL_PATH,
    PREPROCESSING_PIPELINE_PATH,
    os.path.join(
        OUTPUTS_PATH,
        "Final_Random_Forest_Diagnostic_Summary.csv"
    ),
    os.path.join(
        OUTPUTS_PATH,
        "Final_Model_Performance_Comparison.csv"
    )
]

all_files_available = True

for path in required_files:
    exists = os.path.isfile(path)

    print(
        f"{'✓' if exists else '✗'} "
        f"{os.path.basename(path)}"
    )

    if not exists:
        all_files_available = False

# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print("\n" + "=" * 70)

if all_directories_available and all_files_available:
    print("PROJECT ENVIRONMENT VERIFIED SUCCESSFULLY")
    print("Ready to begin the Decision-Support System phase.")
else:
    print("PROJECT ENVIRONMENT VERIFICATION INCOMPLETE")
    print("Review the missing paths/files above before proceeding.")

print("=" * 70)

print("\nProject Root:")
print(PROJECT_ROOT)

ARMY PROVOST DECISION-SUPPORT SYSTEM
PROJECT ENVIRONMENT SETUP
Mounted at /content/drive

Checking project directories...

✓ /content/drive/MyDrive/Army_Provost_ML_Project
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Datasets
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Datasets/Cleaned
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Models
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Figures
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Outputs
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Reports
✓ /content/drive/MyDrive/Army_Provost_ML_Project/Logs

Checking important ML artifacts...

✓ final_random_forest.pkl
✓ preprocessing_pipeline.pkl
✓ Final_Random_Forest_Diagnostic_Summary.csv
✓ Final_Model_Performance_Comparison.csv

PROJECT ENVIRONMENT VERIFIED SUCCESSFULLY
Ready to begin the Decision-Support System phase.

Project Root:
/content/drive/MyDrive/Army_Provost_ML_Project


In [2]:
# ============================================================
# PRIMARY INCIDENT CATEGORY INVENTORY
# DECISION-SUPPORT SYSTEM
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("PRIMARY INCIDENT CATEGORY INVENTORY")
print("=" * 70)

# ------------------------------------------------------------
# Existing project paths
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"

CLEANED_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "Datasets",
    "Cleaned"
)

CLEANED_FILE = os.path.join(
    CLEANED_DATA_PATH,
    "chicago_crimes_cleaned.csv"
)

# ------------------------------------------------------------
# Verify dataset exists
# ------------------------------------------------------------

if not os.path.isfile(CLEANED_FILE):
    raise FileNotFoundError(
        f"Cleaned dataset not found:\n{CLEANED_FILE}"
    )

print("\nDataset found:")
print(CLEANED_FILE)

# ------------------------------------------------------------
# Read ONLY Primary Type
# ------------------------------------------------------------

print("\nReading only the 'Primary Type' column...")
print("The full dataset will NOT be loaded into memory.")

primary_type_series = pd.read_csv(
    CLEANED_FILE,
    usecols=["Primary Type"]
)["Primary Type"]

# ------------------------------------------------------------
# Generate category inventory
# ------------------------------------------------------------

primary_type_counts = (
    primary_type_series
    .value_counts(dropna=False)
    .reset_index()
)

primary_type_counts.columns = [
    "Primary Type",
    "Incident Count"
]

primary_type_counts["Percentage"] = (
    primary_type_counts["Incident Count"]
    / primary_type_counts["Incident Count"].sum()
    * 100
)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CATEGORY INVENTORY")
print("=" * 70)

print(
    f"\nTotal records inspected: "
    f"{len(primary_type_series):,}"
)

print(
    f"Unique Primary Type categories: "
    f"{primary_type_series.nunique(dropna=True)}"
)

print("\nComplete Primary Type Distribution:\n")

display(
    primary_type_counts.round({
        "Percentage": 2
    })
)

# ------------------------------------------------------------
# Save inventory permanently
# ------------------------------------------------------------

OUTPUTS_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs"
)

taxonomy_inventory_path = os.path.join(
    OUTPUTS_PATH,
    "Primary_Type_Category_Inventory.csv"
)

primary_type_counts.to_csv(
    taxonomy_inventory_path,
    index=False
)

print("=" * 70)
print("CATEGORY INVENTORY SAVED")
print("=" * 70)

print("\nSaved to:")
print(taxonomy_inventory_path)

PRIMARY INCIDENT CATEGORY INVENTORY

Dataset found:
/content/drive/MyDrive/Army_Provost_ML_Project/Datasets/Cleaned/chicago_crimes_cleaned.csv

Reading only the 'Primary Type' column...
The full dataset will NOT be loaded into memory.

CATEGORY INVENTORY

Total records inspected: 8,602,734
Unique Primary Type categories: 34

Complete Primary Type Distribution:



,Primary Type,Incident Count,Percentage
0,THEFT,1827377,21.24
1,BATTERY,1567212,18.22
2,CRIMINAL DAMAGE,977357,11.36
3,NARCOTICS,768704,8.94
4,ASSAULT,580094,6.74
5,OTHER OFFENSE,537540,6.25
6,BURGLARY,455436,5.29
7,MOTOR VEHICLE THEFT,444725,5.17
8,DECEPTIVE PRACTICE,400071,4.65
9,ROBBERY,318103,3.70


CATEGORY INVENTORY SAVED

Saved to:
/content/drive/MyDrive/Army_Provost_ML_Project/Outputs/Primary_Type_Category_Inventory.csv


In [4]:
# ============================================================
# CORRECT AND VALIDATE ARMY PROVOST INCIDENT TAXONOMY
# ============================================================

import pandas as pd
from pathlib import Path

print("=" * 70)
print("CORRECTING ARMY PROVOST INCIDENT TAXONOMY")
print("=" * 70)

# ------------------------------------------------------------
# Existing project path
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Army_Provost_ML_Project"
)

OUTPUTS_PATH = PROJECT_ROOT / "Outputs"

taxonomy_path = (
    OUTPUTS_PATH / "Army_Provost_Incident_Taxonomy.csv"
)

# ------------------------------------------------------------
# Corrected taxonomy
# ------------------------------------------------------------

taxonomy_data = [

    [
        "THEFT",
        "Property & Asset Security",
        "Theft / Property Loss",
        "Moderate",
        "Involves unauthorized taking or loss of property or assets."
    ],

    [
        "BATTERY",
        "Personnel & Physical Safety",
        "Physical Assault",
        "High",
        "Involves physical harm or attempted physical harm to a person."
    ],

    [
        "CRIMINAL DAMAGE",
        "Property & Asset Security",
        "Property Damage",
        "Moderate",
        "Involves intentional damage to property or assets."
    ],

    [
        "NARCOTICS",
        "Controlled Substances",
        "Narcotics Activity",
        "High",
        "Involves controlled-substance related activity."
    ],

    [
        "ASSAULT",
        "Personnel & Physical Safety",
        "Assault / Threat of Injury",
        "High",
        "Involves an assault or threat of physical harm."
    ],

    [
        "OTHER OFFENSE",
        "Fraud, Deception & Administrative",
        "Other / Unclassified Offence",
        "Low",
        "Represents offences not captured by the major defined categories."
    ],

    [
        "BURGLARY",
        "Property & Asset Security",
        "Unauthorized Entry / Burglary",
        "Moderate",
        "Involves unlawful entry associated with property-related crime."
    ],

    [
        "MOTOR VEHICLE THEFT",
        "Vehicle & Mobility Security",
        "Vehicle Theft",
        "High",
        "Involves unauthorized taking of a motor vehicle."
    ],

    [
        "DECEPTIVE PRACTICE",
        "Fraud, Deception & Administrative",
        "Fraud / Deception",
        "Moderate",
        "Involves deceptive or fraudulent activity."
    ],

    [
        "ROBBERY",
        "Property & Asset Security",
        "Violent Property Crime",
        "High",
        "Involves property-taking accompanied by force or threat."
    ],

    [
        "CRIMINAL TRESPASS",
        "Property & Asset Security",
        "Unauthorized Access",
        "Moderate",
        "Involves unauthorized entry or presence on property."
    ],

    [
        "WEAPONS VIOLATION",
        "Weapons & Armed Security",
        "Weapons Violation",
        "High",
        "Involves unlawful possession, use, or handling of weapons."
    ],

    [
        "PROSTITUTION",
        "Sexual & Exploitation-Related",
        "Commercial Sexual Activity",
        "Moderate",
        "Represents a category involving commercial sexual activity."
    ],

    [
        "OFFENSE INVOLVING CHILDREN",
        "Child & Vulnerable-Person Safety",
        "Child-Related Offence",
        "Critical",
        "Involves an offence affecting or involving children."
    ],

    [
        "PUBLIC PEACE VIOLATION",
        "Public Order & Discipline",
        "Public Order Violation",
        "Moderate",
        "Involves disruption or violation of public peace and order."
    ],

    [
        "SEX OFFENSE",
        "Sexual & Exploitation-Related",
        "Sexual Offence",
        "High",
        "Involves a sexual offence requiring sensitive handling."
    ],

    [
        "CRIM SEXUAL ASSAULT",
        "Sexual & Exploitation-Related",
        "Sexual Assault",
        "Critical",
        "Involves sexual assault and presents a serious personnel-safety concern."
    ],

    [
        "INTERFERENCE WITH PUBLIC OFFICER",
        "Public Order & Discipline",
        "Authority / Officer Interference",
        "Moderate",
        "Involves interference with an officer or authorized public function."
    ],

    [
        "LIQUOR LAW VIOLATION",
        "Public Order & Discipline",
        "Alcohol-Related Violation",
        "Low",
        "Involves violation of applicable liquor-related regulations."
    ],

    [
        "ARSON",
        "Fire, Destructive & Major Threats",
        "Arson / Fire Threat",
        "High",
        "Involves deliberate fire-setting and potential major property or personnel risk."
    ],

    [
        "GAMBLING",
        "Public Order & Discipline",
        "Gambling Activity",
        "Low",
        "Represents gambling-related activity or violation."
    ],

    [
        "HOMICIDE",
        "Personnel & Physical Safety",
        "Fatal Violence",
        "Critical",
        "Represents fatal violence and an extreme personnel-safety concern."
    ],

    [
        "CRIMINAL SEXUAL ASSAULT",
        "Sexual & Exploitation-Related",
        "Sexual Assault",
        "Critical",
        "Involves serious sexual assault and presents a major personnel-safety concern."
    ],

    [
        "KIDNAPPING",
        "Personnel & Physical Safety",
        "Abduction / Kidnapping",
        "Critical",
        "Involves unlawful abduction or confinement of a person."
    ],

    [
        "STALKING",
        "Personnel & Physical Safety",
        "Persistent Threat / Stalking",
        "High",
        "May indicate persistent targeting or an ongoing threat to an individual."
    ],

    [
        "INTIMIDATION",
        "Personnel & Physical Safety",
        "Threat / Intimidation",
        "High",
        "Involves threats, intimidation, or coercive conduct toward a person."
    ],

    [
        "CONCEALED CARRY LICENSE VIOLATION",
        "Weapons & Armed Security",
        "Unauthorized Weapons Carry",
        "High",
        "Involves violation associated with concealed carrying of a weapon."
    ],

    [
        "OBSCENITY",
        "Public Order & Discipline",
        "Public Conduct Violation",
        "Low",
        "Represents conduct classified as an obscenity-related violation."
    ],

    [
        "PUBLIC INDECENCY",
        "Public Order & Discipline",
        "Public Conduct Violation",
        "Low",
        "Involves conduct considered inappropriate or unlawful in a public setting."
    ],

    [
        "OTHER NARCOTIC VIOLATION",
        "Controlled Substances",
        "Other Drug-Related Violation",
        "High",
        "Represents a narcotics-related violation outside the primary narcotics category."
    ],

    [
        "HUMAN TRAFFICKING",
        "Sexual & Exploitation-Related",
        "Human Trafficking",
        "Critical",
        "Represents severe exploitation involving trafficking of persons."
    ],

    [
        "NON-CRIMINAL",
        "Fraud, Deception & Administrative",
        "Non-Criminal / Administrative",
        "Low",
        "Represents records classified as non-criminal rather than a conventional offence."
    ],

    [
        "RITUALISM",
        "Public Order & Discipline",
        "Unclassified Conduct",
        "Low",
        "Represents a rare category requiring broad operational classification."
    ],

    [
        "DOMESTIC VIOLENCE",
        "Personnel & Physical Safety",
        "Domestic / Interpersonal Violence",
        "High",
        "Involves interpersonal violence and potential personnel-safety concerns."
    ]
]

# ------------------------------------------------------------
# Create DataFrame
# ------------------------------------------------------------

taxonomy_df = pd.DataFrame(
    taxonomy_data,
    columns=[
        "Primary Type",
        "Provost Incident Category",
        "Incident Subcategory",
        "Priority Relevance",
        "Mapping Rationale"
    ]
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

expected_categories = 34

print("\nValidating corrected taxonomy...")
print("-" * 70)

print(f"Total taxonomy rows : {len(taxonomy_df)}")
print(f"Expected categories : {expected_categories}")
print(
    f"Unique Primary Types: "
    f"{taxonomy_df['Primary Type'].nunique()}"
)

duplicate_types = taxonomy_df[
    taxonomy_df["Primary Type"].duplicated(keep=False)
]["Primary Type"].unique()

missing_values = taxonomy_df.isnull().sum()

valid_relevance = {
    "Critical",
    "High",
    "Moderate",
    "Low"
}

invalid_relevance = set(
    taxonomy_df["Priority Relevance"].dropna()
) - valid_relevance

# ------------------------------------------------------------
# Validation results
# ------------------------------------------------------------

print("\nValidation Results")
print("-" * 70)

if len(taxonomy_df) == expected_categories:
    print("✓ Row count validation passed")
else:
    print("✗ Row count validation FAILED")

if taxonomy_df["Primary Type"].nunique() == expected_categories:
    print("✓ Unique category validation passed")
else:
    print("✗ Unique category validation FAILED")

if len(duplicate_types) == 0:
    print("✓ Duplicate category validation passed")
else:
    print("✗ Duplicate categories found:")
    print(duplicate_types)

if missing_values.sum() == 0:
    print("✓ Missing-value validation passed")
else:
    print("✗ Missing values found:")
    print(missing_values[missing_values > 0])

if len(invalid_relevance) == 0:
    print("✓ Priority relevance validation passed")
else:
    print("✗ Invalid relevance levels:")
    print(invalid_relevance)

# ------------------------------------------------------------
# Save corrected taxonomy
# ------------------------------------------------------------

taxonomy_df.to_csv(
    taxonomy_path,
    index=False
)

print("\n" + "=" * 70)
print("CORRECTED TAXONOMY SAVED")
print("=" * 70)

print(f"\nSaved to:")
print(taxonomy_path)

print("\nTaxonomy Preview:")
display(taxonomy_df)

print("\n" + "=" * 70)
print("TAXONOMY VALIDATION COMPLETED")
print("=" * 70)

CORRECTING ARMY PROVOST INCIDENT TAXONOMY

Validating corrected taxonomy...
----------------------------------------------------------------------
Total taxonomy rows : 34
Expected categories : 34
Unique Primary Types: 34

Validation Results
----------------------------------------------------------------------
✓ Row count validation passed
✓ Unique category validation passed
✓ Duplicate category validation passed
✓ Missing-value validation passed
✓ Priority relevance validation passed

CORRECTED TAXONOMY SAVED

Saved to:
/content/drive/MyDrive/Army_Provost_ML_Project/Outputs/Army_Provost_Incident_Taxonomy.csv

Taxonomy Preview:


,Primary Type,Provost Incident Category,Incident Subcategory,Priority Relevance,Mapping Rationale
0,THEFT,Property & Asset Security,Theft / Property Loss,Moderate,Involves unauthorized taking or loss of proper...
1,BATTERY,Personnel & Physical Safety,Physical Assault,High,Involves physical harm or attempted physical h...
2,CRIMINAL DAMAGE,Property & Asset Security,Property Damage,Moderate,Involves intentional damage to property or ass...
3,NARCOTICS,Controlled Substances,Narcotics Activity,High,Involves controlled-substance related activity.
4,ASSAULT,Personnel & Physical Safety,Assault / Threat of Injury,High,Involves an assault or threat of physical harm.
5,OTHER OFFENSE,"Fraud, Deception & Administrative",Other / Unclassified Offence,Low,Represents offences not captured by the major ...
6,BURGLARY,Property & Asset Security,Unauthorized Entry / Burglary,Moderate,Involves unlawful entry associated with proper...
7,MOTOR VEHICLE THEFT,Vehicle & Mobility Security,Vehicle Theft,High,Involves unauthorized taking of a motor vehicle.
8,DECEPTIVE PRACTICE,"Fraud, Deception & Administrative",Fraud / Deception,Moderate,Involves deceptive or fraudulent activity.
9,ROBBERY,Property & Asset Security,Violent Property Crime,High,Involves property-taking accompanied by force ...



TAXONOMY VALIDATION COMPLETED


## Taxonomy-Level Incident Analysis

The validated Army Provost incident taxonomy maps the 34 source
crime categories in the Chicago Crimes dataset into broader
Army Provost-oriented incident categories.

This analysis evaluates how incidents are distributed across these
operational categories and priority levels.

The full cleaned dataset will be processed in a memory-conscious
manner rather than loading unnecessary columns into memory.

In [5]:
# ============================================================
# ARMY PROVOST TAXONOMY-LEVEL INCIDENT ANALYSIS
# ============================================================

import os
import pandas as pd

print("=" * 70)
print("ARMY PROVOST TAXONOMY-LEVEL INCIDENT ANALYSIS")
print("=" * 70)

# ------------------------------------------------------------
# 1. Recreate project paths explicitly
# ------------------------------------------------------------

PROJECT_ROOT = "/content/drive/MyDrive/Army_Provost_ML_Project"

CLEANED_DATA_PATH = os.path.join(
    PROJECT_ROOT,
    "Datasets",
    "Cleaned",
    "chicago_crimes_cleaned.csv"
)

TAXONOMY_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Army_Provost_Incident_Taxonomy.csv"
)

OUTPUT_PATH = os.path.join(
    PROJECT_ROOT,
    "Outputs",
    "Army_Provost_Category_Distribution.csv"
)

# ------------------------------------------------------------
# 2. Verify required files
# ------------------------------------------------------------

print("\nChecking required files...")

if not os.path.exists(CLEANED_DATA_PATH):
    raise FileNotFoundError(
        f"Cleaned dataset not found:\n{CLEANED_DATA_PATH}"
    )

if not os.path.exists(TAXONOMY_PATH):
    raise FileNotFoundError(
        f"Taxonomy file not found:\n{TAXONOMY_PATH}"
    )

print("✓ Cleaned dataset found")
print("✓ Validated taxonomy found")

# ------------------------------------------------------------
# 3. Load taxonomy
# ------------------------------------------------------------

taxonomy_df = pd.read_csv(TAXONOMY_PATH)

print(f"\nTaxonomy rows loaded: {len(taxonomy_df)}")

# ------------------------------------------------------------
# 4. Read only Primary Type from the 8.6M-row dataset
# ------------------------------------------------------------

print("\nReading Primary Type from cleaned dataset...")
print("Only the required column will be loaded.")

primary_type_df = pd.read_csv(
    CLEANED_DATA_PATH,
    usecols=["Primary Type"]
)

print(
    f"Records loaded: {len(primary_type_df):,}"
)

# ------------------------------------------------------------
# 5. Create mapping dictionaries
# ------------------------------------------------------------

category_mapping = dict(
    zip(
        taxonomy_df["Primary Type"],
        taxonomy_df["Provost Incident Category"]
    )
)

priority_mapping = dict(
    zip(
        taxonomy_df["Primary Type"],
        taxonomy_df["Priority Relevance"]
    )
)

# ------------------------------------------------------------
# 6. Map incidents into Army Provost categories
# ------------------------------------------------------------

primary_type_df["Provost Incident Category"] = (
    primary_type_df["Primary Type"]
    .map(category_mapping)
)

primary_type_df["Priority Relevance"] = (
    primary_type_df["Primary Type"]
    .map(priority_mapping)
)

# ------------------------------------------------------------
# 7. Validate mapping coverage
# ------------------------------------------------------------

unmapped_count = (
    primary_type_df["Provost Incident Category"]
    .isna()
    .sum()
)

print("\n" + "=" * 70)
print("MAPPING VALIDATION")
print("=" * 70)

print(
    f"Total incidents       : "
    f"{len(primary_type_df):,}"
)

print(
    f"Unmapped incidents    : "
    f"{unmapped_count:,}"
)

if unmapped_count == 0:
    print("✓ All incidents successfully mapped")
else:
    print("⚠ WARNING: Some incidents were not mapped")

# ------------------------------------------------------------
# 8. Category distribution
# ------------------------------------------------------------

category_distribution = (
    primary_type_df[
        [
            "Provost Incident Category",
            "Priority Relevance"
        ]
    ]
    .value_counts()
    .reset_index(name="Incident Count")
)

category_distribution["Percentage"] = (
    category_distribution["Incident Count"]
    / len(primary_type_df)
    * 100
)

category_distribution = category_distribution.sort_values(
    "Incident Count",
    ascending=False
).reset_index(drop=True)

# ------------------------------------------------------------
# 9. Save results
# ------------------------------------------------------------

category_distribution.to_csv(
    OUTPUT_PATH,
    index=False
)

# ------------------------------------------------------------
# 10. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ARMY PROVOST CATEGORY DISTRIBUTION")
print("=" * 70)

display(
    category_distribution.style.format({
        "Incident Count": "{:,.0f}",
        "Percentage": "{:.2f}%"
    })
)

print("\n" + "=" * 70)
print("RESULT SAVED")
print("=" * 70)

print(f"\nSaved to:")
print(OUTPUT_PATH)

ARMY PROVOST TAXONOMY-LEVEL INCIDENT ANALYSIS

Checking required files...
✓ Cleaned dataset found
✓ Validated taxonomy found

Taxonomy rows loaded: 34

Reading Primary Type from cleaned dataset...
Only the required column will be loaded.
Records loaded: 8,602,734

MAPPING VALIDATION
Total incidents       : 8,602,734
Unmapped incidents    : 0
✓ All incidents successfully mapped

ARMY PROVOST CATEGORY DISTRIBUTION


,Provost Incident Category,Priority Relevance,Incident Count,Percentage
0,Property & Asset Security,Moderate,"3,491,037",40.58%
1,Personnel & Physical Safety,High,"2,159,237",25.10%
2,Controlled Substances,High,"768,871",8.94%
3,"Fraud, Deception & Administrative",Low,"537,568",6.25%
4,Vehicle & Mobility Security,High,"444,725",5.17%
5,"Fraud, Deception & Administrative",Moderate,"400,071",4.65%
6,Property & Asset Security,High,"318,103",3.70%
7,Weapons & Armed Security,High,"130,400",1.52%
8,Public Order & Discipline,Moderate,"76,661",0.89%
9,Sexual & Exploitation-Related,Moderate,"70,523",0.82%



RESULT SAVED

Saved to:
/content/drive/MyDrive/Army_Provost_ML_Project/Outputs/Army_Provost_Category_Distribution.csv
